# BY aarya

In [ ]:
import geopandas as gpd
import folium
from branca.colormap import LinearColormap
import pandas as pd

# Loading the dataset
gdf = gpd.read_file("/content/maharashtra_vru_exposure_output(2)(1).geojson")
gdf = gdf.to_crs(epsg=4326)


# Computing the speed delta
# Converting 'SpeedLimit' to numeric, coercing errors to NaN
gdf['SpeedLimit'] = pd.to_numeric(gdf['SpeedLimit'], errors='coerce')
gdf["speed_delta"] = gdf["F85thPercentileSpeed"] - gdf["SpeedLimit"]

# Colour by VRU_exposure_index (green to red)
colormap = LinearColormap(
    colors=["#1D9E75", "#FAC775", "#EF9F27", "#E24B4A", "#7B0000"],
    vmin=gdf["VRU_exposure_index"].min(),
    vmax=gdf["VRU_exposure_index"].max(),
    caption="VRU Exposure Index (0 = Low risk → 100 = High risk)"
)

# Building the map
m = folium.Map(
    location=[19.7, 75.7],
    zoom_start=7,
    tiles="CartoDB positron"
)

# Adding each segment as a coloured line with tooltip
for _, row in gdf.iterrows():
    geom = row.geometry
    if geom is None:
        continue

    # Extracting coordinates (handling both LineString and MultiLineString)
    if geom.geom_type == "LineString":
        coords = [(lat, lon) for lon, lat in geom.coords]
    elif geom.geom_type == "MultiLineString":
        coords = [(lat, lon) for line in geom.geoms for lon, lat in line.coords]
    else:
        continue

    vru = row["VRU_exposure_index"]
    color = colormap(vru)

    # Line weight: thicker = higher VRU risk
    weight = 2 + (vru / 100) * 4  # ranges 2px to 6px

    # Tooltip shown on hover
    tooltip = folium.Tooltip(f"""
        <b>{row['names_primary'] or 'Unnamed road'}</b><br>
        VRU Exposure: <b>{vru:.1f}</b><br>
        F85 Speed: <b>{row['F85thPercentileSpeed']:.0f} km/h</b><br>
        Speed Limit: <b>{row['SpeedLimit']} km/h</b><br>
        Speed Delta: <b>{row['speed_delta']:+.0f} km/h</b><br>
        Road Class: {row['RoadClass']}<br>
        Land Use: {row['LandUse']}<br>
        POI score: {row['poi_score']:.2f} |
        Crossing: {row['crossing_score']:.2f} |
        Footfall: {row['footfall_score']:.2f}
    """)

    folium.PolyLine(
        locations=coords,
        color=color,
        weight=weight,
        opacity=0.85,
        tooltip=tooltip,
    ).add_to(m)

# Adding colour legend
colormap.add_to(m)

# Adding layer control + title
title_html = """
<div style="position: fixed; top: 10px; left: 50px; z-index: 1000;
     background-color: white; padding: 10px 15px; border-radius: 6px;
     border: 1px solid #ccc; font-family: Arial; font-size: 14px;">
    <b>VRU Exposure Hotspots — Maharashtra</b><br>
    <span style="font-size:11px; color:#555;">
        920 road segments · Colour = VRU risk · Width = risk intensity
    </span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# Saving the files
m.save("vru_exposure_map.html")
print("Saved vru_exposure_map.html")

In [ ]:
# displaying the map
colormap.add_to(m)
m.get_root().html.add_child(folium.Element(title_html))

m

In [ ]:
import folium
import geopandas as gpd
from branca.colormap import StepColormap

gdf_sss = gpd.read_file("/content/maharashtra_sss_scored.geojson")
gdf_sss = gdf_sss.to_crs(epsg=4326)

# Colouring per tier that matches Safe System convention
tier_colors = {
    "Critical": "#7B0000",
    "High":     "#E24B4A",
    "Moderate": "#EF9F27",
    "Low":      "#1D9E75",
}

sss_layer = folium.FeatureGroup(name="Speed Safety Score (SSS)")

for _, row in gdf_sss.iterrows():
    geom = row.geometry
    if geom is None:
        continue

    if geom.geom_type == "LineString":
        coords = [(lat, lon) for lon, lat in geom.coords]
    elif geom.geom_type == "MultiLineString":
        coords = [(lat, lon) for line in geom.geoms for lon, lat in line.coords]
    else:
        continue

    tier = row["SSS_Tier"]
    color = tier_colors.get(tier, "#999999")
    weight = {"Critical": 5, "High": 3.5, "Moderate": 2.5, "Low": 1.5}.get(tier, 2)

    tooltip = folium.Tooltip(f"""
        <b>{row.get('names_primary', 'Unnamed road')}</b><br>
        SSS Score: <b>{row['SSS']:.1f}</b><br>
        Tier: <b>{tier}</b><br>
        F85 Speed: <b>{row['F85thPercentileSpeed']:.0f} km/h</b><br>
        Speed Limit: <b>{row['SpeedLimit']} km/h</b><br>
        Road Class: {row['RoadClass']}<br>
        Land Use: {row['LandUse']}
    """)

    folium.PolyLine(
        locations=coords,
        color=color,
        weight=weight,
        opacity=0.85,
        tooltip=tooltip
    ).add_to(sss_layer)

sss_layer.add_to(m)

In [ ]:
m = folium.Map(
    location=[19.7, 75.7],
    zoom_start=7,
    tiles="CartoDB positron"
)

sss_layer = folium.FeatureGroup(name="Speed Safety Score (SSS)")

for _, row in gdf_sss.iterrows():
    geom = row.geometry
    if geom is None:
        continue

    if geom.geom_type == "LineString":
        coords = [(lat, lon) for lon, lat in geom.coords]
    elif geom.geom_type == "MultiLineString":
        coords = [(lat, lon) for line in geom.geoms for lon, lat in line.coords]
    else:
        continue

    tier = row["SSS_Tier"]
    color = tier_colors.get(tier, "#999999")
    weight = {"Critical": 5, "High": 3.5, "Moderate": 2.5, "Low": 1.5}.get(tier, 2)

    tooltip = folium.Tooltip(f"""
        <b>{row.get('names_primary', 'Unnamed road')}</b><br>
        SSS Score: <b>{row['SSS']:.1f}</b><br>
        Tier: <b>{tier}</b><br>
        F85 Speed: <b>{row['F85thPercentileSpeed']:.0f} km/h</b><br>
        Speed Limit: <b>{row['SpeedLimit']} km/h</b><br>
        Road Class: {row['RoadClass']}<br>
        Land Use: {row['LandUse']}
    """)

    folium.PolyLine(
        locations=coords,
        color=color,
        weight=weight,
        opacity=0.85,
        tooltip=tooltip
    ).add_to(sss_layer)

sss_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m

The most critical roads are where both VRU and SSS are high,because that means:
Lots of pedestrians are near the road and cars going much faster than the posted limit
That combination = someone may die there. That's our Critical-immediate intervention segment. Those are the roads our team is recommending governments fix first

In [ ]:
import folium
import geopandas as gpd
from branca.colormap import LinearColormap

# Map
m = folium.Map(location=[19.7, 75.7], zoom_start=7, tiles="CartoDB positron")

# AYER 1: SSS — red/amber/green by tier
tier_colors  = {"Critical": "#7B0000", "High": "#E24B4A", "Moderate": "#EF9F27", "Low": "#1D9E75"}
tier_weights = {"Critical": 5, "High": 3.5, "Moderate": 2.5, "Low": 1.5}

sss_layer = folium.FeatureGroup(name="🔴 Speed Safety Score (SSS)", show=True)

for _, row in gdf_sss.iterrows():
    geom = row.geometry
    if geom is None: continue
    coords = [(lat, lon) for lon, lat in geom.coords] if geom.geom_type == "LineString" \
             else [(lat, lon) for line in geom.geoms for lon, lat in line.coords]

    tier = row["SSS_Tier"]
    folium.PolyLine(
        locations=coords,
        color=tier_colors.get(tier, "#999"),
        weight=tier_weights.get(tier, 2),
        opacity=0.9,
        tooltip=folium.Tooltip(f"""
            <b>{row.get('names_primary','Unnamed')}</b><br>
            SSS: <b>{row['SSS']:.1f}</b> — <b>{tier}</b><br>
            F85: {row['F85thPercentileSpeed']:.0f} km/h |
            Limit: {row['SpeedLimit']} km/h<br>
            Road: {row['RoadClass']} | {row['LandUse']}
        """)
    ).add_to(sss_layer)

sss_layer.add_to(m)

# LAYER 2: VRU - blue scale
vru_colormap = LinearColormap(
    colors=["#deeaf7", "#6aaed6", "#2171b5", "#08306b"],  # light to dark blue
    vmin=gdf["VRU_exposure_index"].min(),
    vmax=gdf["VRU_exposure_index"].max(),
    caption="VRU Exposure Index (blue scale)"
)

vru_layer = folium.FeatureGroup(name="🔵 VRU Exposure Index", show=False)  # hidden by default

for _, row in gdf.iterrows():
    geom = row.geometry
    if geom is None: continue
    coords = [(lat, lon) for lon, lat in geom.coords] if geom.geom_type == "LineString" \
             else [(lat, lon) for line in geom.geoms for lon, lat in line.coords]

    vru = row["VRU_exposure_index"]
    folium.PolyLine(
        locations=coords,
        color=vru_colormap(vru),
        weight=2 + (vru / 100) * 4,
        opacity=0.9,
        tooltip=folium.Tooltip(f"""
            <b>{row.get('names_primary','Unnamed')}</b><br>
            VRU Index: <b>{vru:.1f}</b><br>
            POI: {row['poi_score']:.2f} |
            Crossing: {row['crossing_score']:.2f} |
            Footfall: {row['footfall_score']:.2f}<br>
            Road: {row['RoadClass']} | {row['LandUse']}
        """)
    ).add_to(vru_layer)

vru_layer.add_to(m)
vru_colormap.add_to(m)

# SSS Legend done manually
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
     background: white; padding: 12px 16px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 12px;">
    <b>Speed Safety Score (SSS)</b><br><br>
    <span style="background:#7B0000;padding:2px 10px;">&nbsp;</span>
    &nbsp;Critical (≥80) — Immediate action<br><br>
    <span style="background:#E24B4A;padding:2px 10px;">&nbsp;</span>
    &nbsp;High (60–79) — Priority review<br><br>
    <span style="background:#EF9F27;padding:2px 10px;">&nbsp;</span>
    &nbsp;Moderate (40–59) — Monitor<br><br>
    <span style="background:#1D9E75;padding:2px 10px;">&nbsp;</span>
    &nbsp;Low (&lt;40) — Acceptable
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

#  Title
title_html = """
<div style="position:fixed; top:10px; left:50px; z-index:1000;
     background:white; padding:10px 15px; border-radius:6px;
     border:1px solid #ccc; font-family:Arial; font-size:13px;">
    <b>Road Speed Safety — Maharashtra</b><br>
    <span style="font-size:11px;color:#555;">
        Toggle layers → SSS (red scale) | VRU Exposure (blue scale)
    </span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

folium.LayerControl(collapsed=False).add_to(m)

m